In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [26]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_core.prompts import PromptTemplate

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [4]:
len(docs)

9

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap=200 )

In [6]:
splitter_docs= splitter.split_documents(docs)

In [7]:
len(splitter_docs)

26

In [8]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [9]:
from openai import vector_stores


vector_stores=InMemoryVectorStore.from_documents(
    documents=splitter_docs,
    embedding=embeddings
)

### agent = tools, llm, prompt

In [10]:
similer_record = vector_stores.similarity_search("Patients Name")

In [16]:
similer_record[0].page_content

'Report Status    \nFemale\n27 Years:\n:\n:\n:\nAge\nGender\nReported        \nP\n9/7/2025   4:56:00PM\nDR NITIN NAHAR\n474764803\nMs. NIKITA  CHUDHARY:\n:\n:\n:\n:\nName        \nLab No.    \nRef By \nCollected       \nA/c Status \n10/7/2025  6:31:50PM\nFinal\nLPL - Bhopal Lab II\nPlot No.05, Mandakini Housing Society, \nNear Apurti Shopping Mall, Kolar Main \nRoad, Bhopal, M.P. -  462042\n::Collected at            Processed at             BHOPAL CC-82\nMr Rachel V John Pata So Vitus John Mig 26 \nGraund,Indrapuri, Phone: 8770817968\n \nTest Report      \nTest Name Results Units Bio. Ref. Interval\n4. The concentration of Vitamin B12 obtained with different assay methods cannot be used \ninterchangeably due to differences in assay methods and reagent specificity\nVITAMIN D, 25 - HYDROXY, SERUM\n(ECLIA)\nnmol/L 75.00 - 250.0067.50\nInterpretation \n -------------------------------------------------------------\n| LEVEL         | REFERENCE RANGE | COMMENTS                  |'

In [36]:
from langchain_core.tools import tool

@tool
def retriever_tool(query: str):
    """
    This tool retrieves relevant data from the medical PDF report.
    """
    print("Tool Called:", query)

    docs = vector_stores.similarity_search(query=query, k=4)
    context = ""

    for doc in docs:
        context += doc.page_content + "\n\n"

    return context.strip()


In [37]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [38]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [39]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [40]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called: patient name doctor name


In [41]:
result = response["messages"][-1].content

In [42]:
print(result)

[{'type': 'text', 'text': 'Based on the medical report:\n\n* **Patient Name:** Ms. Nikita Choudhary\n* **Doctors / Medical Professionals Mentioned:**\n  * **Referring Physician:** Dr. Nitin Nahar ("Ref By")\n  * **Technical Director - Flowcytometry:** Dr. Beena Chandrasekhar\n  * **Consultant - Hemato-Oncopathology & Flowcytometry:** Dr. Nagarjun Sai Jaine\n  * **Sr. Consultant Pathologist - Hematology & Immunology:** Dr. Sunanda', 'extras': {'signature': 'EmAKXgFpFH0TJcsPtQf8AsEw5uZ9n3jmRgMxZhLc8mBxHGDwnlru9lAGMGP+X6kko4sN1u36DxyfpAhSnBZJwlYIuV3XalGnmxvkbkIT7pUaM8X0pwAoUIhYGYT0r0fL1g0='}}]
